[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinculum3141-ship-it/bike-availability-data-science-full/blob/main/notebooks/Module_02_Data_Acquisition/M2_01_amsterdam_bike_api.ipynb)

# 🚴 Module 02: Amsterdam Bike Data Acquisition

**Purpose**: Learn to fetch real-time bike availability data from the CityBikes API  
**Module**: Module 02 - Data Acquisition  
**Author**: [Your Name]  
**Date**: 2026-01-15

---

## 📋 Overview

In this notebook, you will:
- [ ] Understand the CityBikes API structure
- [ ] Fetch real-time bike availability data for Amsterdam
- [ ] Parse and explore JSON responses
- [ ] Implement error handling for API requests
- [ ] Save raw data with proper documentation
- [ ] Validate data quality

**Goal**: Build confidence fetching data from REST APIs!

**Estimated Time**: 90-120 minutes

---

## 🎓 Learning Path Note

**This Module**: Focus on exploring and understanding data acquisition concepts using notebook-based experiments. Write code directly in cells to learn.

**Learning Strategy**: 
- ✅ DO: Experiment and learn in this notebook
- ⏸️ LATER: Productionize code into reusable scripts (Module 8)

---

---

## 📖 Part 1: Understanding the API

### What is CityBikes API?

[CityBikes](http://api.citybik.es/v2/) is a **free, open API** that aggregates bike-sharing data from cities worldwide.

**Key Features**:
- ✅ **No authentication required** - just make HTTP requests
- ✅ **Real-time data** - updated every few minutes
- ✅ **Global coverage** - 600+ cities
- ✅ **JSON format** - easy to parse with Python

### API Endpoints

**Base URL**: `http://api.citybik.es/v2/`

| Endpoint | Description | Example |
|----------|-------------|---------|
| `/networks` | List all available networks | `GET /v2/networks` |
| `/networks/{id}` | Get specific network details | `GET /v2/networks/ov-fiets` |

### Netherlands Bike Network

For this project, we'll use the **OV-fiets** network:
- **Network ID**: `ov-fiets` (OV-fiets bike sharing across the Netherlands)
- **Coverage**: All of the Netherlands, including Amsterdam stations
- **Note**: You'll filter for Amsterdam stations in your analysis

> 💡 **Why OV-fiets?** This network provides reliable, consistent data for learning purposes. In Module 1, you learned this endpoint works well with the CityBikes API.

### Data Structure

The API returns JSON with this structure:
```json
{
  "network": {
    "id": "ov-fiets",
    "name": "OV-fiets",
    "location": {
      "city": "",
      "country": "NL",
      "latitude": 52.3676,
      "longitude": 4.9041
    },
    "stations": [
      {
        "id": "station-001",
        "name": "Amsterdam Centraal",
        "latitude": 52.3791,
        "longitude": 4.9003,
        "free_bikes": 5,
        "empty_slots": 15,
        "timestamp": "2026-01-15T10:30:00.000000Z"
      }
    ]
  }
}
```

**Key Fields**:

- `free_bikes`: Number of bikes available to rent- `timestamp`: When the data was collected
- `empty_slots`: Number of empty docking slots

---

## 🔧 Part 2: Setup

Run this cell first to set up the environment.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 1. IMPORTS
# ═══════════════════════════════════════════════════════════

# Standard library
import os
import sys
import json
from datetime import datetime
from pathlib import Path

# Third-party imports
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import seaborn as sns

# Configure pandas display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Check if running in Google Colab
if 'google.colab' in sys.modules:
    print("📍 Running in Google Colab")
    # Uncomment to clone repository
    # !git clone https://github.com/vinculum3141-ship-it/bike-availability-data-science-full.git
    # %cd bike-availability-data-science-full
    # !pip install -q -r requirements.txt
else:
    print("📍 Running locally")

# Add project root to path
project_root = os.path.abspath('../..' if 'notebooks' in os.getcwd() else '.')
if project_root not in sys.path:
    sys.path.append(project_root)

print("✅ Setup complete!")
print(f"📁 Working directory: {os.getcwd()}")
print(f"🐍 Python version: {sys.version.split()[0]}")
print(f"📅 Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## ⚙️ Part 3: Configuration

Define API endpoints and file paths.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 2. CONFIGURATION
# ═══════════════════════════════════════════════════════════

# API Configuration
BASE_URL = "http://api.citybik.es/v2"
NETWORK_ID = "ov-fiets"  # OV-fiets bike sharing across Netherlands
NETWORK_URL = f"{BASE_URL}/networks/{NETWORK_ID}"

# Note: OV-fiets covers all Netherlands - you'll filter for Amsterdam in analysis

# File paths
DATA_DIR = Path('../../data/raw') if 'notebooks' in os.getcwd() else Path('data/raw')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Output filename with timestamp
TIMESTAMP = datetime.now().strftime('%Y-%m-%d_%H%M%S')
OUTPUT_FILE = DATA_DIR / f'amsterdam_bike_{TIMESTAMP}.json'

print("✅ Configuration set")
print(f"🌐 API URL: {NETWORK_URL}")
print(f"💾 Output file: {OUTPUT_FILE}")

---

## 🌐 Part 4: Fetch Data from API (Apply Your M1 Knowledge!)

### 🧠 Task 1: Make Your First API Request

**📚 What You Already Know:**

In [M1_03_open_data_sources.ipynb](../Module_01_Introduction/M1_03_open_data_sources.ipynb), you learned:
- How to make API requests with `requests.get()`
- How to check status codes (200 = success)
- How to parse JSON responses with `.json()`
- Basic error checking

**🎯 Your Task:**

Fetch bike data from the CityBikes API using the skills from Module 1.

**Requirements:**
1. Make a GET request to `NETWORK_URL` (defined above)
2. Check if the response status code is 200
3. Parse the JSON response
4. Print:
   - Network name
   - Location (city, country)
   - Number of stations

**💡 Hints:**
- Start with `response = requests.get(...)`
- Check success: `if response.status_code == 200:`
- Parse JSON: `data = response.json()`
- Access nested data: `data['network']['name']`
- Count stations: `len(data['network']['stations'])`

**⏱️ Estimated Time:** 5-10 minutes

**❓ Stuck?** Review [M1_03](../Module_01_Introduction/M1_03_open_data_sources.ipynb) or check the solutions notebook.

---

In [ ]:
# ═══════════════════════════════════════════════════════════
# 3. TASK 1: FETCH BIKE DATA (8-12 lines of code)
# ═══════════════════════════════════════════════════════════

"""
Complete this task using what you learned in Module 1!

Steps:
1. Make GET request to NETWORK_URL
2. Store response in a variable
3. Check if status code is 200
4. If successful, parse JSON
5. Print network name, location, and station count
6. Handle the case when request fails

Remember: You did this exact pattern in M1_03!
"""

print("📡 Fetching bike data from CityBikes API...")
print(f"🔗 URL: {NETWORK_URL}\n")

# Your code here (8-12 lines)

### ✅ Verify Your Work

After completing Task 1 above:
1. ✅ Make sure you can see the network name and station count
2. ✅ Check that `data` is a dictionary with a 'network' key
3. ✅ Verify status code is 200

**Check Solution:** If you're stuck or want to verify, see `M2_01_amsterdam_bike_api_SOLUTIONS.ipynb`

---

### 🔍 Task 2: Explore the JSON Structure

Now that you have the data, let's understand its structure.

**Your Task:**
1. Get the first station from `data['network']['stations']`
2. Print it in a readable format (use `json.dumps()` with `indent=2`)
3. Loop through the station's keys and print each field name and its type

**💡 Hints:**
- First station: `first_station = data['network']['stations'][0]`
- Pretty print: `json.dumps(first_station, indent=2)`  
- Field types: `type(value).__name__`

**⏱️ Estimated Time:** 5 minutes

In [ ]:
# ═══════════════════════════════════════════════════════════
# 4. TASK 2: EXPLORE JSON STRUCTURE (5-8 lines of code)
# ═══════════════════════════════════════════════════════════

"""
Explore the structure of a single station's data.

Steps:
1. Extract the first station from data['network']['stations']
2. Print it using json.dumps with nice formatting
3. Print each field name and its type
"""

# Your code here (5-8 lines)

---

## ⚠️ Part 5: Error Handling (Build Production-Ready Code)

### 🧠 Task 3: Implement Robust Error Handling

**Why This Matters:**
- APIs can fail (timeouts, server errors, network issues)
- Production code must handle failures gracefully
- Users need clear error messages

**Your Task:**

Create a function `fetch_bike_data_safe()` that fetches bike data with proper error handling.

**Function Requirements:**
```python
def fetch_bike_data_safe(network_id, timeout=10):
    """
    Fetch bike data with comprehensive error handling.
    
    Parameters:
    -----------
    network_id : str
        The network ID (e.g., 'ns-bikes')
    timeout : int
        Request timeout in seconds (default: 10)
    
    Returns:
    --------
    dict or None
        JSON data if successful, None if any error occurs
    """
```

**What to Handle:**
1. **Timeouts**: `requests.exceptions.Timeout`
2. **Connection Errors**: `requests.exceptions.ConnectionError`
3. **HTTP Errors** (4xx, 5xx): `requests.exceptions.HTTPError`
4. **General Request Errors**: `requests.exceptions.RequestException`
5. **JSON Parsing Errors**: `json.JSONDecodeError`

**💡 Hints:**
- Build URL from `BASE_URL` and `network_id`
- Use `try/except` blocks for each error type
- Use `response.raise_for_status()` to catch HTTP errors
- Print helpful error messages
- Return `None` on any failure

**⏱️ Estimated Time:** 15-20 minutes

**📚 Review M1 if needed:** M1_03 showed basic try/except patterns

---

In [ ]:
# ═══════════════════════════════════════════════════════════
# 5. TASK 3: ERROR HANDLING (20-30 lines of code)
# ═══════════════════════════════════════════════════════════

"""
Create a production-ready function with comprehensive error handling.

Structure:
1. Define function with parameters
2. Build the URL
3. Create try block with request logic
4. Add except blocks for each error type (5 total)
5. Test the function with valid and invalid inputs

Remember: Each except block should:
  - Print a user-friendly error message
  - Return None
"""

def fetch_bike_data_safe(network_id, timeout=10):
    """
    Fetch bike data with error handling.
    
    Parameters:
    -----------
    network_id : str
        The network ID (e.g., 'ns-bikes')
    timeout : int
        Request timeout in seconds
    
    Returns:
    --------
    dict or None
        JSON data if successful, None otherwise
    """
    # Your code here (20-30 lines)
    # Hint: Start with url = f"{BASE_URL}/networks/{network_id}"
    pass


# Test your function
print("🧪 Testing error handling function...")
print("=" * 60)

# Test 1: Valid network
print("\nTest 1: Valid network")
result = fetch_bike_data_safe(NETWORK_ID, timeout=10)
if result:
    print(f"✅ Success! Fetched {len(result['network']['stations'])} stations")

# Test 2: Invalid network (should handle gracefully)
print("\nTest 2: Invalid network ID")
result = fetch_bike_data_safe("fake-network-12345", timeout=10)
if result is None:
    print("✅ Correctly handled invalid network")

print("\n" + "=" * 60)

> **💡 Stuck?** Check `M2_01_amsterdam_bike_api_SOLUTIONS.ipynb` for complete implementations after you've tried yourself!

---


In [ ]:
import time

def fetch_with_retry(url, max_retries=3, timeout=10):
    """
    Fetch data with exponential backoff retry logic.
    
    Parameters:
    -----------
    url : str
        URL to fetch
    max_retries : int
        Maximum number of retry attempts
    timeout : int
        Request timeout in seconds
    
    Returns:
    --------
    dict or None
        JSON data if successful, None if all retries failed
    """
    for attempt in range(max_retries):
        try:
            print(f"🔄 Attempt {attempt + 1}/{max_retries}...")
            response = requests.get(url, timeout=timeout)
            response.raise_for_status()
            print("✅ Success!")
            return response.json()
            
        except requests.exceptions.HTTPError as e:
            # Don't retry client errors (4xx)
            if 400 <= e.response.status_code < 500:
                print(f"❌ Client error {e.response.status_code}: {e}")
                return None
            # Retry server errors (5xx)
            print(f"⚠️ Server error {e.response.status_code}: {e}")
            
        except requests.exceptions.Timeout:
            print(f"⚠️ Timeout after {timeout} seconds")
            
        except requests.exceptions.ConnectionError as e:
            print(f"⚠️ Connection error: {e}")
            
        except requests.exceptions.RequestException as e:
            print(f"⚠️ Unexpected error: {e}")
        
        # Exponential backoff: wait 2^attempt seconds
        if attempt < max_retries - 1:
            wait_time = 2 ** attempt
            print(f"⏳ Waiting {wait_time} seconds before retry...\n")
            time.sleep(wait_time)
    
    print("❌ All retries failed")
    return None

# Test it!
print("Testing retry logic with bad URL:")
test_url = "http://api.citybik.es/v2/networks/fake-network-12345"
result = fetch_with_retry(test_url, max_retries=3)

---

## Part 5a: Retry Logic & Rate Limiting

When working with external APIs in production, you need to handle two common challenges:

1. **Temporary Failures**: APIs can be temporarily unavailable or slow
2. **Rate Limits**: APIs restrict how many requests you can make

Let's implement production-ready patterns for both.

---

### Pattern 1: Retry Logic with Exponential Backoff

**Key Concepts:**
- **Exponential Backoff**: Wait progressively longer between retries (1s, 2s, 4s, 8s...)
- **Selective Retry**: Only retry on errors that might resolve (5xx server errors, timeouts)
- **Max Retries**: Limit attempts to avoid infinite loops

**When to Retry:**
- ✅ 5xx Server Errors (temporary server issues)
- ✅ Network timeouts
- ✅ Connection errors
- ❌ 4xx Client Errors (your request is wrong - retry won't help!)


### 🎯 Best Practices Summary

**Retry Logic:**
- ✅ Retry on temporary failures (timeouts, 5xx errors)
- ❌ Don't retry on client errors (4xx - it won't help!)
- ✅ Use exponential backoff (1s, 2s, 4s, 8s...)
- ✅ Limit max retries (3-5 is reasonable)
- ✅ Log each attempt for debugging

**Rate Limiting:**
- ✅ Add delays between bulk requests (1-2 seconds)
- ✅ Check API documentation for limits
- ✅ Consider using `time.sleep()` or `asyncio` for delays
- ✅ Be respectful of free APIs - they're a gift!

**Production Tips:**
- Use `requests.Session()` for connection pooling
- Consider `backoff` library for complex retry logic
- Implement circuit breakers for persistent failures
- Monitor and log API usage patterns

---

### 💡 When to Use These Patterns

**Use Retry Logic When:**
- Calling external APIs (especially free/public ones)
- Network conditions are unreliable
- Building production data pipelines
- You need resilient data collection

**Use Rate Limiting When:**
- Fetching from multiple endpoints
- API has documented rate limits
- You're making many requests in a loop
- Building automated/scheduled jobs

---

In [ ]:
# ═══════════════════════════════════════════════════════════
# 5B. RATE LIMITING FOR BULK REQUESTS
# ═══════════════════════════════════════════════════════════

def fetch_multiple_networks_safe(network_ids, delay=1.0):
    """
    Fetch data for multiple networks with rate limiting.
    
    Parameters:
    -----------
    network_ids : list
        List of network IDs to fetch
    delay : float
        Seconds to wait between requests
    
    Returns:
    --------
    dict
        Network ID -> data mapping
    """
    import time
    
    results = {}
    
    for i, network_id in enumerate(network_ids):
        url = f"{BASE_URL}/networks/{network_id}"
        
        print(f"\n📡 Fetching {network_id} ({i+1}/{len(network_ids)})...")
        
        data = fetch_with_retry(url, max_retries=2)
        
        if data:
            results[network_id] = data
            print(f"✅ Fetched {len(data['network']['stations'])} stations")
        else:
            print(f"❌ Failed to fetch {network_id}")
        
        # Rate limiting: wait before next request (except after last one)
        if i < len(network_ids) - 1:
            print(f"⏳ Rate limit delay: {delay}s...")
            time.sleep(delay)
    
    return results


# Example: Fetch data for multiple Dutch bike networks
dutch_networks = ['ov-fiets']  # Using working endpoint from M1

print("Fetching multiple networks with rate limiting:")
print("=" * 60)
network_data = fetch_multiple_networks_safe(dutch_networks, delay=1.0)

print(f"\n📊 Summary:")
print(f"   Total networks fetched: {len(network_data)}")
for network_id, data in network_data.items():
    print(f"   • {network_id}: {len(data['network']['stations'])} stations")

### 🧠 Pattern 2: Rate Limiting for Bulk Requests

When fetching data from multiple endpoints, add delays to avoid rate limits.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 5A. RETRY LOGIC WITH EXPONENTIAL BACKOFF
# ═══════════════════════════════════════════════════════════

def fetch_with_retry(url, max_retries=3, timeout=10):
    """
    Fetch data with retry logic and exponential backoff.
    
    Parameters:
    -----------
    url : str
        URL to fetch
    max_retries : int
        Maximum number of retry attempts
    timeout : int
        Request timeout in seconds
    
    Returns:
    --------
    dict or None
        JSON data if successful, None otherwise
    """
    import time
    
    for attempt in range(max_retries):
        try:
            print(f"🔄 Attempt {attempt + 1}/{max_retries}...")
            response = requests.get(url, timeout=timeout)
            response.raise_for_status()
            
            print(f"✅ Success on attempt {attempt + 1}!")
            return response.json()
        
        except requests.exceptions.Timeout:
            print(f"⏱️ Timeout on attempt {attempt + 1}")
            
        except requests.exceptions.HTTPError as e:
            # Don't retry on 4xx errors (client errors)
            if 400 <= e.response.status_code < 500:
                print(f"❌ Client error {e.response.status_code} - not retrying")
                return None
            print(f"⚠️ Server error on attempt {attempt + 1}: {e}")
            
        except requests.exceptions.RequestException as e:
            print(f"⚠️ Request failed on attempt {attempt + 1}: {e}")
        
        # Exponential backoff: wait 2^attempt seconds before retry
        if attempt < max_retries - 1:  # Don't sleep after last attempt
            wait_time = 2 ** attempt
            print(f"⏳ Waiting {wait_time} seconds before retry...")
            time.sleep(wait_time)
    
    print("❌ All retry attempts failed")
    return None


# Test the retry logic
print("Testing retry logic with valid endpoint:")
print("=" * 60)
data = fetch_with_retry(NETWORK_URL, max_retries=3)

if data:
    print(f"\n✅ Successfully fetched data for {data['network']['name']}")
else:
    print("\n❌ Failed to fetch data after all retries")

---

## 🔄 Part 6: Retry Logic & Rate Limiting (Production Patterns)

### Why Retry Logic Matters

**Real-World API Challenges:**
- Network hiccups cause temporary failures
- APIs occasionally timeout or return 5xx errors
- Rate limits require waiting between requests

**Solution**: Implement smart retry logic with exponential backoff

---

### 📚 Key Concepts

**Exponential Backoff:**
- First retry: Wait 1 second
- Second retry: Wait 2 seconds  
- Third retry: Wait 4 seconds
- Prevents hammering failing servers

**Rate Limiting:**
- Add delays between requests
- Respect API quotas
- Be a good API citizen

---

### 🧠 Pattern 1: Simple Retry with Exponential Backoff

**Note:** The examples below show retry patterns for reference. Your Task 3 function doesn't need retry logic.

---

## 📊 Part 7: Convert to DataFrame

### 🧠 Task 4.1: Extract and Convert to DataFrame

**What You Know from Module 1:**
You practiced extracting nested data and creating DataFrames in M1_04. The CityBikes API returns data like:
```python
{
  "network": {
    "stations": [
      {"id": "...", "name": "...", "latitude": ..., ...},
      {"id": "...", "name": "...", "latitude": ..., ...}
    ]
  }
}
```

**Your Task:**
1. Extract the stations list from the nested JSON structure
2. Convert to a pandas DataFrame
3. Display the shape, first 3 rows, and column info

**Validation:** Your DataFrame should have 400+ rows and 10+ columns.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 6. TASK 4.1: EXTRACT AND CONVERT TO DATAFRAME
# ═══════════════════════════════════════════════════════════

# TODO: Extract stations list and convert to DataFrame
# Write your code here



### 🧠 Task 4.2: Clean Column Names and Parse Timestamps

**Your Task:**
1. Parse the `timestamp` column to datetime using `pd.to_datetime()`
2. Rename columns for clarity:
   - `free_bikes` → `bikes_available`
   - `empty_slots` → `docks_available`

**Hint:** Use `df.rename(columns={...})` for renaming.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 7. TASK 4.2: CLEAN COLUMN NAMES AND PARSE TIMESTAMPS
# ═══════════════════════════════════════════════════════════

# TODO: Parse timestamp and rename columns
# Write your code here


### 🧠 Task 4.3: Add Derived Columns

**Your Task:** Create three new calculated columns:
1. `total_capacity` = bikes_available + docks_available
2. `utilization_pct` = (bikes_available / total_capacity × 100), rounded to 2 decimals
3. `is_empty` = True if bikes_available == 0, else False

**Hint:** Use vectorized operations: `df['new_col'] = df['col1'] + df['col2']`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 8. TASK 4.3: ADD DERIVED COLUMNS
# ═══════════════════════════════════════════════════════════

# TODO: Create calculated columns
# Write your code here


### 🧠 Task 4.4: Add Network Metadata

**Your Task:** Add these columns from the `data` JSON response:
- `network_id` (from `data['network']['id']`)
- `network_name` (from `data['network']['name']`)
- `city` (from `data['network']['location']['city']`)
- `country` (from `data['network']['location']['country']`)

**Hint:** Assign scalar values to create columns: `df['new_col'] = value`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 9. TASK 4.4: ADD NETWORK METADATA
# ═══════════════════════════════════════════════════════════

# TODO: Add metadata columns
# Write your code here


### 🧠 Task 4.5: Reorder and Validate

**Your Task:**
1. Reorder columns to this order:
   ```python
   ['id', 'name', 'latitude', 'longitude',
    'bikes_available', 'docks_available', 'total_capacity', 'utilization_pct',
    'timestamp', 'network_id', 'network_name', 'city', 'country']
   ```
2. Display first 10 rows and summary statistics for numeric columns

**Hint:** Use `df = df[column_list]` to reorder columns.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 10. TASK 4.5: REORDER AND VALIDATE
# ═══════════════════════════════════════════════════════════

# TODO: Reorder columns and display results
# Write your code here



---

## 🔍 Part 8: Data Quality and Exploration

Now let's explore the bike availability patterns!

### 🧠 Task 5.1: Calculate Summary Statistics

**Your Task:** Calculate and display these statistics:
1. Use `.describe()` on numeric columns: bikes_available, docks_available, total_capacity, utilization_pct
2. Calculate and print:
   - Total number of stations
   - Total bikes available across all stations
   - Total docks available
   - Average station capacity
   - Average utilization percentage
3. Identify problem stations:
   - Count stations with 0 bikes (empty)
   - Count stations with 0 docks (full)

**Hint:** Use `.sum()`, `.mean()`, and boolean filtering `df[df['column'] == value]`

In [ ]:
# ═══════════════════════════════════════════════════════════
# TASK 5.1: CALCULATE SUMMARY STATISTICS
# ═══════════════════════════════════════════════════════════

# TODO: Display summary statistics and identify patterns
# Write your code here


### 🧠 Task 5.2: Create 4-Panel Visualization Dashboard

- Add mean line: `axvline(mean_value, color='red', linestyle='--')`

**Your Task:** Create a 2x2 subplot figure showing:- For top 10: `df.nlargest(n, 'column')`

- Access subplots with `axes[row, col]`

**Panel 1 (Top-Left):** Histogram of bikes_available- Use `fig, axes = plt.subplots(2, 2, figsize=(15, 10))`

- 20 bins, blue color, with mean line**Hints:**



**Panel 2 (Top-Right):** Histogram of utilization_pct- 2 bars with values labeled on top

- 20 bins, orange color, with mean line**Panel 4 (Bottom-Right):** Bar chart comparing average bikes_available vs docks_available



**Panel 3 (Bottom-Left):** Horizontal bar chart of top 10 stations by bikes_available- Show station names on y-axis

In [ ]:
# ═══════════════════════════════════════════════════════════
# TASK 5.2: CREATE 4-PANEL VISUALIZATION DASHBOARD
# ═══════════════════════════════════════════════════════════

# TODO: Create 4-panel visualization
# Write your code here


### 🧠 Task 5.3: Create Geographic Station Map

**Your Task:** Create a scatter plot map of station locations with these requirements:

1. Plot stations using longitude (x-axis) and latitude (y-axis)
2. Color points by `bikes_available` (use colormap 'RdYlGn' - red=low, green=high)
3. Size points by `total_capacity` (multiply by 5 for visibility)
4. Add colorbar with label
5. Add grid, title, and axis labels
6. **Bonus:** Annotate the largest station by capacity

**Hints:**
- `plt.scatter(x, y, c=color_values, s=size_values, cmap='RdYlGn')`
- `plt.colorbar(scatter, label='...')`
- Find largest: `df.nlargest(1, 'column').iloc[0]`

In [ ]:
# ═══════════════════════════════════════════════════════════
# TASK 5.3: CREATE GEOGRAPHIC STATION MAP
# ═══════════════════════════════════════════════════════════

# TODO: Create geographic scatter plot
# Write your code here



### 🧠 Task 5.4: Custom Analysis (Your Choice!)

**Your Task:** Create ONE additional analysis or visualization that answers an interesting question about the data.

**Ideas:**
- Which stations have the highest/lowest utilization?
- Is there a relationship between station capacity and location?
- What percentage of stations are problematic (empty or full)?
- Create a different type of visualization (box plot, violin plot, etc.)

**Requirements:**
- Include a clear title explaining what you're showing
- Add appropriate labels and formatting
- Write 1-2 sentences interpreting your findings

In [ ]:
# ═══════════════════════════════════════════════════════════
# TASK 5.4: CUSTOM ANALYSIS
# ═══════════════════════════════════════════════════════════

# TODO: Your custom analysis here
# Write your code here



---

## 🔥 Part 9: Optional Advanced Challenges

**For advanced learners**: These challenges push beyond the basics. Try at least ONE!

### Challenge 6.1: Build a Retry Decorator 🔄

**Task:** Create a Python decorator that automatically retries API calls on failure with exponential backoff.

**Requirements:**
- Decorator `@retry_on_failure(max_retries=3, backoff_factor=2)`
- Retry only on timeout/connection errors (not 404)
- Wait time: backoff_factor^(attempt) seconds
- Log each attempt

**Hint:** Use `functools.wraps` and recursion or loops

### Challenge 6.2: Real-time Monitoring Function 📊

**Task:** Create a function that fetches bike data every N seconds and plots real-time changes.

**Requirements:**
- Function signature: `monitor_bikes(network_id, duration_seconds, interval=60)`
- Display live updating plot of total bikes available
- Use `plt.ion()` for interactive plotting
- Gracefully handle KeyboardInterrupt (Ctrl+C)

### Challenge 6.3: Multi-Network Comparison 🌍

**Task:** Fetch data from 3 different bike networks (e.g., Amsterdam, London, New York) and create a comparison dashboard.

**Requirements:**
- Fetch from 3 networks using the function you created
- Compare: total stations, avg bikes/station, avg utilization
- Create a 3-panel visualization comparing the networks
- Save combined data to single CSV

**Hint:** Loop through network IDs: `['velib', 'santander-cycles', 'citi-bike-nyc']`

### Challenge 6.4: API Response Caching 💾

**Task:** Implement a caching system to avoid redundant API calls.

**Requirements:**
- Cache API responses in memory with timestamp
- Only re-fetch if cache is older than N minutes
- Function: `get_bike_data_cached(network_id, cache_duration=5)`
- Display "Using cached data" or "Fetching new data" message

**Hint:** Use a global dictionary with network_id as key


In [ ]:
# ═══════════════════════════════════════════════════════════
# PART 6: OPTIONAL ADVANCED CHALLENGES
# ═══════════════════════════════════════════════════════════

# TODO: Implement one or more advanced challenges here
# Write your code here


---

## ✅ Part 10: Data Validation

Let's validate our data quality before saving.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 11. DATA VALIDATION
# ═══════════════════════════════════════════════════════════

def validate_bike_data(df):
    """
    Validate bike availability data quality.
    
    Returns:
    --------
    bool : True if all validations pass
    """
    validations = []
    
    # Check 1: Required columns exist
    required_cols = ['id', 'name', 'bikes_available', 'timestamp']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        print(f"❌ Missing columns: {missing_cols}")
        validations.append(False)
    else:
        print(f"✅ All required columns present")
        validations.append(True)
    
    # Check 2: No negative values
    if (df['bikes_available'] < 0).any():
        print(f"❌ Found negative bike counts")
        validations.append(False)
    else:
        print(f"✅ No negative bike counts")
        validations.append(True)
    
    # Check 3: Timestamps are valid
    if df['timestamp'].isna().any():
        print(f"❌ Found missing timestamps")
        validations.append(False)
    else:
        print(f"✅ All timestamps present")
        validations.append(True)
    
    # Check 4: Coordinates are valid
    valid_lat = df['latitude'].between(-90, 90).all()
    valid_lon = df['longitude'].between(-180, 180).all()
    if not (valid_lat and valid_lon):
        print(f"❌ Invalid coordinates found")
        validations.append(False)
    else:
        print(f"✅ All coordinates valid")
        validations.append(True)
    
    # Check 5: No duplicates
    if df.duplicated(subset=['id']).any():
        print(f"❌ Found duplicate station IDs")
        validations.append(False)
    else:
        print(f"✅ No duplicate stations")
        validations.append(True)
    
    # Check 6: Reasonable data ranges
    if df['bikes_available'].max() > 100:
        print(f"⚠️ Warning: Unusually high bike count ({df['bikes_available'].max()})")
    
    return all(validations)


# Run validation
print("🔍 Validating data quality...")
print("=" * 60)
is_valid = validate_bike_data(df_bikes)
print("=" * 60)

if is_valid:
    print("✅ All validation checks passed!")
else:
    print("❌ Some validation checks failed - review data before saving")

---

## 💾 Part 11: Save Raw Data

Save both the raw JSON response and the cleaned DataFrame.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 12. SAVE RAW DATA
# ═══════════════════════════════════════════════════════════

# Save raw JSON response
print("💾 Saving raw JSON data...")
with open(OUTPUT_FILE, 'w') as f:
    json.dump(data, f, indent=2)
print(f"✅ Saved raw JSON: {OUTPUT_FILE}")
print(f"📦 File size: {OUTPUT_FILE.stat().st_size / 1024:.1f} KB")

# Also save as CSV for easy viewing
csv_file = OUTPUT_FILE.with_suffix('.csv')
df_bikes.to_csv(csv_file, index=False)
print(f"✅ Saved CSV: {csv_file}")
print(f"📦 File size: {csv_file.stat().st_size / 1024:.1f} KB")

# Create metadata file
metadata = {
    'source': 'CityBikes API',
    'api_url': NETWORK_URL,
    'network_id': NETWORK_ID,
    'network_name': data['network']['name'],
    'fetch_timestamp': datetime.now().isoformat(),
    'num_stations': len(df_bikes),
    'total_bikes_available': int(df_bikes['bikes_available'].sum()),
    'data_timestamp': df_bikes['timestamp'].iloc[0].isoformat(),
    'city': data['network']['location']['city'],
    'country': data['network']['location']['country']
}

metadata_file = OUTPUT_FILE.with_suffix('.metadata.json')
with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✅ Saved metadata: {metadata_file}")

print("\n" + "=" * 60)
print("📊 Data Acquisition Summary:")
print("=" * 60)
print(f"Stations fetched: {len(df_bikes)}")
print(f"Total bikes available: {df_bikes['bikes_available'].sum()}")
print(f"Files saved:")
print(f"  • {OUTPUT_FILE.name}")
print(f"  • {csv_file.name}")
print(f"  • {metadata_file.name}")

---

## 📝 Part 12: Summary

### What You've Learned ✅

In this notebook, you:
1. ✅ Connected to the CityBikes REST API
2. ✅ Implemented error handling for API requests
3. ✅ Parsed JSON responses and converted to DataFrames
4. ✅ Performed data cleaning and transformation
5. ✅ Validated data quality
6. ✅ Visualized bike availability patterns
7. ✅ Saved raw data with proper documentation

### Key Takeaways 💡

1. **APIs are just URLs** that return structured data (usually JSON)
2. **Error handling is essential** - networks fail, APIs change, timeouts happen
3. **Always validate data** before saving or analyzing
4. **Document your data sources** - future you will thank current you!
5. **Save raw data** before any transformations (immutable source of truth)

### Data Files Created 📁

- `data/raw/amsterdam_bike_{timestamp}.json` - Raw API response
- `data/raw/amsterdam_bike_{timestamp}.csv` - Cleaned DataFrame
- `data/raw/amsterdam_bike_{timestamp}.metadata.json` - Data documentation

### Next Steps 🚀

Now that you have bike data, proceed to:
- **M2_02_weather_data_api.ipynb** - Fetch weather data
- **M2_03_data_storage.ipynb** - Learn storage best practices
- **M2_04_merge_datasets.ipynb** - Combine bike and weather data

### 🧠 Reflection Questions

1. **What challenges might arise** from fetching real-time data at different times of day?
2. **How would you handle API rate limits** if you needed to fetch data every minute?
3. **What other validation checks** could you add to ensure data quality?
4. **How could you automate** this data collection process?

**Write your reflections below** ⬇️

### My Reflections

[Your thoughts here]

---

## 📚 References

- [CityBikes API Documentation](http://api.citybik.es/v2/)
- [Requests Library Documentation](https://requests.readthedocs.io/)
- [Pandas I/O Tools](https://pandas.pydata.org/docs/user_guide/io.html)
- [REST API Tutorial](https://restfulapi.net/)
- [JSON Format Guide](https://www.json.org/json-en.html)

---

**🎉 Congratulations!** You've successfully completed M2_01 - Amsterdam Bike Data Acquisition!